In [3]:
from utils_shiprocket import prepare_data

train_df,val_df,test_df = prepare_data(train_examples=None)
print(len(train_df), len(val_df), len(test_df))

Resolving data files:   0%|          | 0/83 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/83 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/82 [00:00<?, ?it/s]

62247 7781 7780


In [8]:
import json

# 1. Extract the columns into a dictionary using zip
id_to_endpoint = dict(zip(train_df["id"], train_df["endpoint_bool"]))

# 2. Save the dictionary to a JSON file
with open("id_to_endpoint_train.json", "w", encoding="utf-8") as f:
    json.dump(id_to_endpoint, f, indent=4)

print(f"Successfully saved {len(id_to_endpoint)} entries to id_to_endpoint_trian.json")

Successfully saved 62247 entries to id_to_endpoint.json


In [9]:
import json

# 1. Extract the columns into a dictionary using zip
id_to_endpoint = dict(zip(val_df["id"], val_df["endpoint_bool"]))

# 2. Save the dictionary to a JSON file
with open("id_to_endpoint_val.json", "w", encoding="utf-8") as f:
    json.dump(id_to_endpoint, f, indent=4)

print(f"Successfully saved {len(id_to_endpoint)} entries to id_to_endpoint_val.json")

Successfully saved 7781 entries to id_to_endpoint.json


In [2]:
train_pd = train_df.remove_columns(["audio"]).to_pandas() # for memory saving , future EDA

In [3]:
train_pd

,id,language,endpoint_bool,midfiller,endfiller,synthetic,spoken_text,dataset
0,f193b823-1ae3-4deb-876b-2ba5d95fc2e1,pol,False,False,True,True,None,chirp3_1
1,7ab1837a-b20d-46d5-b49a-496557e7c8b2,ben,False,False,True,True,None,chirp3_2
2,19ab9171-a452-4332-a890-30d08185677b,rus,False,True,True,True,None,chirp3_1
3,f09739bf-9949-48e6-bdf3-1e66c041567f,vie,False,False,True,True,None,chirp3_2
4,9eec974c-b882-44f1-9d43-e2136c69ced5,eng,True,None,None,False,None,liva_1
...,...,...,...,...,...,...,...,...
216762,8a6e99db-a4de-43dc-ade9-17c08cdbd090,deu,True,False,False,True,None,chirp3_1
216763,0a666258-df73-4936-ae70-4be184700af0,eng,False,None,None,False,None,liva_1
216764,8c4130fc-94eb-48b8-a125-2b82de41ec5d,nor,True,False,False,True,None,chirp3_2
216765,f567ac5c-9be2-40bb-9bb3-def8b0934972,nor,True,False,False,True,None,chirp3_2


In [4]:
def rms_threshold_baseline(y, sr = 22050, energy_threshold=0.01, min_silence_duration=0.7, frame_length=2048, hop_length=512):
    rms = librosa.feature.rms(y=y, frame_length=frame_length, hop_length=hop_length)[0]
    is_silent = rms < energy_threshold  # boolean array per frame

    # Count consecutive silent frames from the end
    silent_frames = 0
    for val in is_silent[::-1]:  # walk backwards from the end of the clip
        if val:
            silent_frames += 1
        else:
            break

    frame_duration = hop_length / sr
    trailing_silence_sec = silent_frames * frame_duration

    return trailing_silence_sec >= min_silence_duration # True = this is a turn , false = user is still speaking ( not a turn )

In [5]:
import io
import librosa
from tqdm import tqdm
y_pred = []
y_true = []

for item in tqdm(val_df):
    audio_info = item['audio']
    y, sr = librosa.load(io.BytesIO(audio_info['bytes']), sr=22050)
    y_pred.append(rms_threshold_baseline(y, sr))
    y_true.append(item['endpoint_bool'])

print(len(y_pred), len(y_true))

100%|██████████| 27096/27096 [01:29<00:00, 302.43it/s]

27096 27096


In [6]:
import utils_shiprocket

baseline_scores = utils_shiprocket.evaluate_binary_classifier(y_true, y_pred)
baseline_scores

{'accuracy': 0.5084514319456747,
 'precision': 0.5570652173913043,
 'recall': 0.07576317540099047,
 'f1_score': 0.13338538616695947}

In [14]:
import os
import io
import soundfile as sf
from tqdm import tqdm

out_dir = "val_samples"
os.makedirs(out_dir, exist_ok=True)

n_samples = 1000  # change as needed

for i in tqdm(range(min(n_samples, len(val_df)))):
    example = val_df[i]
    audio = example['audio']
    label = int(example['endpoint_bool'])  # 0 or 1

    # use the original filename (without extension) if available, else fallback to index
    if audio.get('path'):
        base_name = os.path.splitext(os.path.basename(audio['path']))[0]
    else:
        base_name = f"sample_{i}"

    out_path = os.path.join(out_dir, f"{base_name}_label_{label}.wav")

    data, samplerate = sf.read(io.BytesIO(audio['bytes']))
    sf.write(out_path, data, samplerate)

    print(f"Saved: {out_path}")

print(f"\nDone — {n_samples} files written to ./{out_dir}")

  2%|▏         | 20/1000 [00:00<00:10, 96.67it/s]

Saved: val_samples/c392b019-8df6-4a50-95d1-558f572f3fe0_label_0.wav
Saved: val_samples/b832db69-5852-42cf-a7b8-dc945e4c551d_label_1.wav
Saved: val_samples/68691ccc-7624-4862-a71d-bae7d8cbb8ee_label_0.wav
Saved: val_samples/2ba30d30-099a-42cb-a5f3-d78078ef8fd5_label_0.wav
Saved: val_samples/4a7d094a-b215-4dfb-ab4d-202f0ce01356_label_1.wav
Saved: val_samples/94a25d5a-b3ba-41e2-9556-6f30c23260bd_label_0.wav
Saved: val_samples/8adf4ded-a064-4970-b856-ab9c09ec46b2_label_0.wav
Saved: val_samples/d94877f4-3ea3-49e2-b0c3-17a9ef995d59_label_0.wav
Saved: val_samples/31518025-83b0-4cbf-a67e-5bcef3fe995c_label_1.wav
Saved: val_samples/5168e93d-cbf9-4272-9861-e23af5aba84e_label_1.wav
Saved: val_samples/f5603fbf-0796-499a-9b54-df8c953d846e_label_0.wav
Saved: val_samples/07c1eb69-0836-4779-bbe5-243a92b59965_label_1.wav
Saved: val_samples/f5dad778-f7d4-4f35-8627-f2db2694e08a_label_1.wav
Saved: val_samples/8fdd778f-a933-4b8e-9d4e-92c626437f7c_label_0.wav
Saved: val_samples/e4f7f498-a052-4093-866a-426a6

  3%|▎         | 30/1000 [00:00<00:11, 84.66it/s]

Saved: val_samples/ba12a59c-9bc7-444c-811f-74a1305328ba_label_0.wav
Saved: val_samples/8446ee08-d8f6-4876-9eda-1b8327c98eb4_label_0.wav
Saved: val_samples/27534177-b5cb-4ea3-a59a-52c0599a9c2c_label_0.wav
Saved: val_samples/9b273538-1505-4b5c-9dfd-cf843a9f4827_label_0.wav
Saved: val_samples/bfcd0c6e-4a7f-49c2-91c2-822bfa5dd70e_label_0.wav
Saved: val_samples/227a8d4d-4589-4e78-9fdf-5967377f6bef_label_0.wav
Saved: val_samples/ec4831a2-6f41-4e29-8a3d-9d07eb5ac220_label_1.wav
Saved: val_samples/2b301aaa-0d48-452a-886d-5eac1f27c667_label_0.wav
Saved: val_samples/3feb9922-cd4e-41ea-a85e-0dad55fbafa9_label_1.wav
Saved: val_samples/05316707-8a0f-42cb-addd-716fb0c5b4a9_label_0.wav
Saved: val_samples/18e02b0b-4b82-48a4-beca-e557b0bb4e66_label_1.wav
Saved: val_samples/81541a96-5c8f-415a-bec6-1fdd2a96d66d_label_0.wav
Saved: val_samples/f4cebd56-bca2-4113-a072-16cc1ebdb461_label_0.wav
Saved: val_samples/3a1d55c7-c2cc-41ed-b17e-54c6e6a26ba1_label_0.wav
Saved: val_samples/a8c6f5e7-8870-4601-90ed-8026f

  6%|▌         | 60/1000 [00:00<00:07, 120.80it/s]

...

Saved: val_samples/7498875b-899c-401a-842d-0bcec9262423_label_0.wav
Saved: val_samples/0645fce0-0e17-461c-bf88-bb726dc1249a_label_1.wav
Saved: val_samples/f6363fc5-8e72-49f6-926e-db7a5f25edf1_label_0.wav
Saved: val_samples/e2c02d7a-c8f9-48fd-925e-733f1b341616_label_0.wav
Saved: val_samples/680258b1-158f-48cb-a206-f39a8a58d312_label_0.wav
Saved: val_samples/914915bd-8516-4e24-b294-f43debba2d66_label_0.wav
Saved: val_samples/c6d02ac3-a351-46a5-9281-dd5f28c9dc85_label_0.wav
Saved: val_samples/2affec7e-21c5-45e9-8177-10c6c9036e0d_label_1.wav
Saved: val_samples/ca02d3f3-edc6-4741-a122-2488b84a8d68_label_0.wav
Saved: val_samples/84e649ee-9379-4936-b018-aefeb84bbd5a_label_0.wav
Saved: val_samples/f22eff76-edf8-4b7e-ae9a-7c712e6f4937_label_1.wav
Saved: val_samples/f92eceed-900f-4f0f-86b3-7362c3ec923d_label_1.wav
Saved: val_samples/3462a871-fc3e-40bc-98eb-031f5bc9bd72_label_0.wav
Saved: val_samples/1a9ade7e-8e63-47db-9cc2-ecc8888156a1_label_0.wav
Saved: val_samples/3f2d4f7f-9dba-4431-9426-09a38

 97%|█████████▋| 968/1000 [00:06<00:00, 165.04it/s]

Saved: val_samples/1748074e-b481-445f-bda6-8b7389e84dad_label_0.wav
Saved: val_samples/42db14a8-e7f9-409c-9810-26c00bb971f7_label_1.wav
Saved: val_samples/65dff6ff-dff9-4116-8aad-17c40ec35abf_label_0.wav
Saved: val_samples/ddfb1d2e-7aa1-42a3-b864-b09feea95932_label_0.wav
Saved: val_samples/52cc5a3c-f543-4de6-ae7b-17179725908c_label_1.wav
Saved: val_samples/a99379b5-1f0e-4fae-b4fe-8729ebcdee2f_label_0.wav
Saved: val_samples/4b0c0953-9c39-4ca9-8c67-b34ce5746099_label_0.wav
Saved: val_samples/3708d98c-c767-4ee4-8af6-4107cd8c2c23_label_0.wav
Saved: val_samples/3a5c1895-0399-492a-8d1a-da8be270a3be_label_1.wav
Saved: val_samples/9506d7e4-e7bf-4ce8-a267-3b1490d3f257_label_0.wav
Saved: val_samples/c90f9056-7419-4489-9700-a9cff7bb38c1_label_0.wav
Saved: val_samples/4fd36c63-a8f3-4708-baf8-ea882caf0f4c_label_0.wav
Saved: val_samples/4a3bf319-6139-40cf-9843-a600d08c80b7_label_1.wav
Saved: val_samples/4f2b22bc-5f11-4c75-812b-8fee1e9e7ed2_label_0.wav
Saved: val_samples/f3de0961-cc9d-4571-a75f-b95a1

100%|██████████| 1000/1000 [00:06<00:00, 150.40it/s]

Saved: val_samples/29750a4a-b51a-4f3e-b287-b3869ea1a27e_label_1.wav
Saved: val_samples/c1e8683a-e52e-4c98-851c-ec94e54ef864_label_1.wav
Saved: val_samples/6540a652-71bc-417d-9634-5f4cb51e7c23_label_0.wav
Saved: val_samples/a4fe82e3-9fb1-4559-81b5-6fc26000319b_label_0.wav
Saved: val_samples/169639ca-8d29-4a76-875f-e5cd9d80f763_label_0.wav
Saved: val_samples/498416cc-e102-4315-a5f0-fb599e4d3b22_label_1.wav
Saved: val_samples/13293576-2b4d-4428-b29a-c6c304f1fd4b_label_0.wav
Saved: val_samples/95b59a71-810b-4539-a442-1bea3a2a2149_label_0.wav
Saved: val_samples/83df6c33-c0d2-4af9-98d8-ab1f46b7c55e_label_0.wav
Saved: val_samples/7b234ea9-aefb-4ad1-8698-c5c8406f6b9b_label_1.wav
Saved: val_samples/a10649cb-11a7-4f58-ae90-c6b8192a79e4_label_0.wav
Saved: val_samples/ce45cb23-85b4-4348-83ce-116fd95d81b0_label_1.wav
Saved: val_samples/6ff501c7-5a6a-4549-92f0-87584a1cb4c1_label_0.wav
Saved: val_samples/31a9edda-0d6b-4b9d-b6a6-a11ad91c8885_label_1.wav
Saved: val_samples/27d2761f-63e6-48f8-9e32-b5948